# 📊 Análise Comparativa de Candidaturas por Gênero no Estado de SP
### Projeto de TCC - Estado de São Paulo

Este notebook realiza a análise comparativa da distribuição e evolução de candidaturas por **gênero (homens e mulheres)** nas eleições municipais no Estado de São Paulo, utilizando os microdados abertos do TSE.

---

### 🎯 Objetivos Principais:
1. **Comparação Geral**: Quantificar a evolução absoluta e percentual de homens e mulheres a cada ano eleitoral.
2. **Cota de Gênero (Lei nº 9.504/1997)**: Avaliar a aderência à exigência de no mínimo 30% de candidaturas de cada gênero ao longo do tempo (destaque para a evolução após a obrigatoriedade estipulada em 2009 / eleições 2012).
3. **Recorte por Cargo**: Contrastar o Legislativo (*Vereador*) com o Executivo (*Prefeito* e *Vice-Prefeito*).
4. **Taxa de Sucesso / Eleitos**: Analisar a proporção de mulheres que conseguiram se eleger em comparação com a proporção de candidatas.

In [ ]:
# 1. Importação de bibliotecas e configuração do ambiente
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Garantir acesso ao módulo src
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

try:
    from src.config import YEARS, RAW_DIR, TARGET_UF
except ImportError:
    YEARS = [2012, 2016, 2020, 2024]
    RAW_DIR = BASE_DIR / "data" / "raw"
    TARGET_UF = "SP"

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["figure.dpi"] = 120

# Paleta padronizada para gênero
PALETTE_GENERO = {
    "FEMININO": "#e75480",
    "MASCULINO": "#2b5c8f",
    "NÃO DIVULGÁVEL": "#888888",
    "NÃO INFORMADO": "#aaaaaa"
}

print(f"✓ Ambiente configurado! Anos configurados: {YEARS} (UF: {TARGET_UF})")

## 📁 2. Carregamento e Consolidação dos Dados
Carregamos os dados de candidaturas para SP de todos os anos selecionados, filtrando as colunas essenciais para maximizar a performance e minimizar o uso de memória.

In [ ]:
# Colunas essenciais para a análise
COLS = ["ANO_ELEICAO", "SG_UF", "NM_UE", "CD_CARGO", "DS_CARGO", "DS_GENERO", "DS_SIT_TOT_TURNO"]

dfs = []
for ano in sorted(YEARS):
    csv_path = RAW_DIR / f"consulta_cand_{ano}" / f"consulta_cand_{ano}_{TARGET_UF}.csv"
    if not csv_path.exists():
        print(f"⚠️ Arquivo não encontrado: {csv_path.name} (Pule se não foi baixado)")
        continue

    print(f"Lendo: {csv_path.name}...", end=" ", flush=True)
    df_ano = pd.read_csv(
        csv_path,
        sep=";",
        encoding="latin1",
        usecols=COLS,
        dtype={
            "ANO_ELEICAO": "int64",
            "SG_UF": "str",
            "NM_UE": "str",
            "DS_CARGO": "str",
            "DS_GENERO": "str",
            "DS_SIT_TOT_TURNO": "str"
        },
        low_memory=False
    )
    # Padronização de textos
    df_ano["DS_GENERO"] = df_ano["DS_GENERO"].fillna("NÃO INFORMADO").str.upper().str.strip()
    df_ano["DS_CARGO"] = df_ano["DS_CARGO"].fillna("").str.upper().str.strip()
    dfs.append(df_ano)
    print(f"({len(df_ano):,} registros)")

df_cand = pd.concat(dfs, ignore_index=True)
print(f"\n✓ Total de candidaturas consolidadas: {len(df_cand):,}")
df_cand.head()

## 📈 3. Panorama Geral: Homens vs. Mulheres por Ano
Abaixo geramos as tabelas de contagem absoluta e percentual por gênero a cada pleito eleitoral.

In [ ]:
# Tabela de Frequência Absoluta
tab_abs = pd.crosstab(df_cand["ANO_ELEICAO"], df_cand["DS_GENERO"], margins=True, margins_name="Total")
print("--- Quantidade Absoluta de Candidatos por Ano e Gênero ---")
display(tab_abs)

# Tabela de Frequência Relativa (%)
tab_pct = pd.crosstab(df_cand["ANO_ELEICAO"], df_cand["DS_GENERO"], normalize="index") * 100
print("\n--- Proporção (%) de Candidatos por Ano e Gênero ---")
display(tab_pct.round(2))

In [ ]:
# Gráfico de Barras: Comparação de Volume Absoluto
df_plot = df_cand[df_cand["DS_GENERO"].isin(["MASCULINO", "FEMININO"])].copy()
contagem = df_plot.groupby(["ANO_ELEICAO", "DS_GENERO"]).size().reset_index(name="QTD")

plt.figure(figsize=(11, 6))
ax = sns.barplot(
    data=contagem,
    x="ANO_ELEICAO",
    y="QTD",
    hue="DS_GENERO",
    palette=PALETTE_GENERO
)

# Adicionar valores formatados acima das barras
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{int(height):,}",
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom',
            fontsize=9.5, fontweight='bold', xytext=(0, 4),
            textcoords='offset points'
        )

anos_str = ", ".join(map(str, sorted(df_cand["ANO_ELEICAO"].unique())))
plt.title(f"Volume Absoluto de Candidaturas por Gênero em {TARGET_UF} ({anos_str})", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Ano da Eleição", fontweight='bold')
plt.ylabel("Quantidade de Candidatos", fontweight='bold')
plt.ylim(0, contagem["QTD"].max() * 1.15)
plt.legend(title="Gênero")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de Evolução Proporcional (%) com a Linha da Cota de 30%
cols_mf = [c for c in ["MASCULINO", "FEMININO"] if c in tab_pct.columns]
tab_pct_mf = tab_pct[cols_mf].copy()

# Recalcular proporcional apenas entre Masculino e Feminino para visualização clara de 100%
tab_pct_mf_norm = tab_pct_mf.div(tab_pct_mf.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 6))
tab_pct_mf_norm.plot(kind="bar", stacked=True, color=[PALETTE_GENERO[c] for c in cols_mf], ax=ax, width=0.55)

# Linha da Cota Mínima (70% Homens / 30% Mulheres)
ax.axhline(70, color="#333333", linestyle="--", linewidth=1.8, label="Piso Legal de 30% Mulheres (70% Homens)")

# Rótulos dentro das barras
for i, (ano, row) in enumerate(tab_pct_mf_norm.iterrows()):
    pct_masc = row["MASCULINO"]
    pct_fem = row["FEMININO"]
    ax.text(i, pct_masc / 2, f"{pct_masc:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=11)
    ax.text(i, pct_masc + pct_fem / 2, f"{pct_fem:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.title(f"Evolução da Proporção de Gênero (%) nas Candidaturas em {TARGET_UF}", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Ano da Eleição", fontweight='bold')
plt.ylabel("Percentual (%)", fontweight='bold')
plt.ylim(0, 105)
plt.xticks(rotation=0)
plt.legend(title="Gênero", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 🏛️ 4. Distribuição por Cargo: Executivo vs. Legislativo
A cota de 30% incide sobre as candidaturas proporcionais (*Vereador*). Vejamos como se comporta a distribuição no Executivo (*Prefeito* e *Vice-Prefeito*).

In [ ]:
# Percentual de mulheres dentro de cada cargo por ano
prop_fem_cargo = (
    df_cand[df_cand["DS_GENERO"] == "FEMININO"].groupby(["DS_CARGO", "ANO_ELEICAO"]).size() / 
    df_cand.groupby(["DS_CARGO", "ANO_ELEICAO"]).size() * 100
).reset_index(name="PCT_FEMININO")

print("--- Percentual de Mulheres Candidatas por Cargo e Ano (%): ---")
tab_cargo_pivot = prop_fem_cargo.pivot(index="DS_CARGO", columns="ANO_ELEICAO", values="PCT_FEMININO")
display(tab_cargo_pivot.round(2))

# Gráfico comparativo
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=prop_fem_cargo,
    x="DS_CARGO",
    y="PCT_FEMININO",
    hue="ANO_ELEICAO",
    palette="Blues"
)

ax.axhline(30, color="#e75480", linestyle="--", linewidth=1.8, label="Cota Proporcional Mínima (30%)")

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{height:.1f}%",
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom',
            fontsize=9, fontweight='bold', xytext=(0, 4),
            textcoords='offset points'
        )

plt.title(f"Representatividade Feminina (% Mulheres) por Cargo Eletivo em {TARGET_UF}", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Cargo Eletivo", fontweight='bold')
plt.ylabel("% Mulheres Candidatas", fontweight='bold')
plt.ylim(0, max(45, prop_fem_cargo["PCT_FEMININO"].max() * 1.15))
plt.legend(title="Ano da Eleição", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 🏆 5. Taxa de Sucesso: Candidatas vs. Candidatas Eleitas
A presença de mulheres nas urnas reflete na proporção de mulheres eleitas?

In [ ]:
# Identificação de candidatos eleitos (abrange nomenclaturas históricas do TSE de 2004 a 2024)
STATUS_ELEITO = [
    "ELEITO",
    "ELEITO POR QP",
    "ELEITO POR MÉDIA",
    "ELEITO POR MDIA",
    "MÉDIA",
    "MEDIA"
]
df_cand["ST_ELEITO"] = df_cand["DS_SIT_TOT_TURNO"].fillna("").str.upper().str.strip().isin(STATUS_ELEITO)
df_eleitos = df_cand[df_cand["ST_ELEITO"]].copy()

# Proporção de mulheres candidatas vs eleitas
pct_cand_fem = (df_cand[df_cand["DS_GENERO"] == "FEMININO"].groupby("ANO_ELEICAO").size() / df_cand.groupby("ANO_ELEICAO").size() * 100)
pct_eleit_fem = (df_eleitos[df_eleitos["DS_GENERO"] == "FEMININO"].groupby("ANO_ELEICAO").size() / df_eleitos.groupby("ANO_ELEICAO").size() * 100)

df_comparativo = pd.DataFrame({
    "% Mulheres Candidatas": pct_cand_fem,
    "% Mulheres Efetivamente Eleitas": pct_eleit_fem
})

print("--- Comparativo de Representatividade: Candidatas vs. Eleitas ---")
display(df_comparativo.round(2))

# Gráfico comparativo
ax = df_comparativo.plot(kind="bar", figsize=(11, 6), color=["#e75480", "#3a86ff"], width=0.6)

for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{height:.1f}%",
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom',
            fontsize=9.5, fontweight='bold', xytext=(0, 4),
            textcoords='offset points'
        )

plt.title(f"Funil de Representatividade: % Candidatas vs. % Eleitas ({TARGET_UF})", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Ano da Eleição", fontweight='bold')
plt.ylabel("Percentual (%)", fontweight='bold')
plt.ylim(0, max(45, df_comparativo.max().max() * 1.2))
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()